# Lab 3: Train and compare ACT

**Goal:** Train ACT on a fixed task, reload its checkpoint, and compare one change in demonstration count with one change in action chunk size. A classical controller also needs a fixed test set and a record of each failed run before you compare settings.

**Inputs:** The pinned bowl-plate demonstrations, two camera videos, the fixed development states, and the local ACT recipe. **Compute:** A local GPU and EGL simulator. **Cloud cost:** US$0 for this prepared local run. **Disk:** Allow several GiB for data, checkpoints, and videos. **Time:** 3–5 learner hours; the measured job run time is in the saved reference report.

Run from the repository root. The equivalent CLI command for the full reference is `.venv/bin/python -m vla_lab.experiments.act --config configs/train/act_task.yaml --output .local-data/runs/my-act-suite`. For a long run, use `robot-lab jobs submit` as shown in the [Lab 3 guide](README.md).


## 1. Read the declared recipe

The first two variants use 10 and 39 complete training episodes. Both predict 20 actions. The third variant uses the same 39 episodes and predicts 10 actions. All three execute at most 10 actions before they ask for a new chunk. Each variant has its own training statistics. The fixed 20 development states are separate from the demonstration validation episodes. Final simulator states 20–49 stay closed during tuning.


In [ ]:
from pathlib import Path
import json
import sys
import time
import tempfile
import yaml
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display
from vla_lab.experiments.act import load_act_suite_config

def find_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository.")


WORKSPACE = find_root()
assert (WORKSPACE / "configs" / "train" / "act_task.yaml").is_file(), "The required course file is missing."
DATA_ROOT = WORKSPACE / ".local-data"
CONFIG_PATH = WORKSPACE / "configs" / "train" / "act_task.yaml"
recipe = load_act_suite_config(CONFIG_PATH, WORKSPACE / "configs")
print("Task:", recipe.task_alias, "seed:", recipe.seed, "steps per variant:", recipe.steps)
print("Development states:", recipe.state_ids)
print("Variants:", [(v.name, v.episode_count, v.chunk_size) for v in recipe.variants])

## 2. Submit a short training job and reconnect

This one-step probe checks the actual training entry point and saved job controls. The worker has a run ID, heartbeat, log, and cancel command. Repeating this cell after a kernel restart reconnects to the same completed job. A full reference run trains 200 steps per variant; the probe is not a useful policy.


In [ ]:
from vla_lab.notebooks import jobs

command = [
    sys.executable, "-m", "vla_lab.training.act",
    "--task", recipe.task_alias,
    "--steps", "1", "--batch-size", str(recipe.batch_size),
    "--checkpoint-every", "1", "--seed", str(recipe.seed),
    "--output", "{run_dir}/training",
]
probe = jobs.submit(
    config={"lab": 3, "kind": "one_step_act", "recipe_sha256": recipe.config_sha256},
    purpose="lab3-training-entry-v1", seed=recipe.seed, command=command, workspace=WORKSPACE,
)
RUN_ID = probe["run_id"]
print("Run ID:", RUN_ID, "phase:", probe["phase"])
for _ in range(480):
    state = jobs.status(RUN_ID, WORKSPACE)
    if state["phase"] in ("succeeded", "failed", "cancelled", "interrupted"):
        break
    time.sleep(0.25)
assert state["phase"] == "succeeded", f"Training job did not finish: {state}; {jobs.logs(RUN_ID, WORKSPACE)}"
print("Final phase:", state["phase"])
print("Saved files:", len(jobs.manager(WORKSPACE).artifacts(RUN_ID)))
print("Log tail:", jobs.logs(RUN_ID, WORKSPACE, lines=5))

## 3. Resume from a complete checkpoint

A resume run restores weights, optimizer state, and PyTorch random state. It writes a new run directory. The fixed sampler names each batch from the seed and step, so the resumed path can match an uninterrupted path. The supplied ACT reference bundles lack saved random state and cannot give this exact continuation.


In [ ]:
from vla_lab.training.act import train_act_reference
from vla_lab.policy.bundle import load_act_bundle

source = Path(state["run_dir"]) / "training" / "checkpoint-000001"
resume_root = DATA_ROOT / "runs"
resume_root.mkdir(parents=True, exist_ok=True)
resume_parent = Path(tempfile.mkdtemp(prefix="lab3-resume-", dir=resume_root))
continued = train_act_reference(
    task_alias=recipe.task_alias,
    configs_root=WORKSPACE / "configs",
    data_root=DATA_ROOT,
    output_dir=resume_parent / "training",
    steps=2,
    batch_size=recipe.batch_size,
    checkpoint_every=1,
    device=recipe.device,
    seed=recipe.seed,
    resume_from=source,
)
checkpoint = continued.output_dir / "checkpoint-000002"
policy, preprocessor, postprocessor, manifest = load_act_bundle(checkpoint)
print("Continued step:", continued.steps, "loss:", continued.last_loss)
print("Checkpoint task:", manifest["task_alias"], "split hash:", manifest["split_sha256"])
print("Saved processor steps:", len(preprocessor.steps), len(postprocessor.steps))

## 4. Inspect the full reference and its failures

The prepared suite trained each variant from a new initialization with the same seed and step budget. It evaluated the same 20 development states for each setting. Read the saved per-state rows; a failure remains in the denominator. The plots below use the exact saved training curves and Wilson intervals. These are three recipe outcomes from one training seed, not a general ranking of methods.


In [ ]:
SUITE_DIR = DATA_ROOT / "runs" / "act-lab3-reference-v1"
summary = json.loads((SUITE_DIR / "summary.json").read_text())
assert summary["config_sha256"] == recipe.config_sha256
assert summary["state_ids"] == list(recipe.state_ids)
assert summary["partition"] == "development" and not summary["final_states_used"]
print("Valid for comparison:", summary["valid_for_comparison"])
for row in summary["variants"]:
    print(row["variant"], "episodes:", row["train_episode_count"],
          "frames:", row["train_frame_count"], "chunk:", row["chunk_size"],
          "development:", f"{row['development_successes']}/{row['development_attempts']}",
          "errors:", row["policy_errors"], row["infrastructure_errors"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for row in summary["variants"]:
    path = SUITE_DIR / "variants" / row["variant"] / "training" / "metrics.jsonl"
    metrics = [json.loads(line) for line in path.read_text().splitlines()]
    axes[0].plot([item["step"] for item in metrics], [item["loss"] for item in metrics], label=row["variant"])
axes[0].set(xlabel="Training step", ylabel="ACT loss", title="One training seed")
axes[0].legend()
labels = [row["variant"] for row in summary["variants"]]
values = [row["development_successes"] / row["development_attempts"] for row in summary["variants"]]
intervals = [row["development_wilson_95"] for row in summary["variants"]]
errors = [[value - interval[0] for value, interval in zip(values, intervals)],
          [interval[1] - value for value, interval in zip(values, intervals)]]
axes[1].bar(labels, values, yerr=errors, capsize=4)
axes[1].set(ylabel="Development success fraction", ylim=(0, 1), title="Same 20 simulator states")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 5. Compare paired state outcomes

The variants used the same ordered simulator states. A paired report counts states where only one policy succeeds and gives a 95% interval for each success fraction. Its exact McNemar value is a small-sample check, not proof that a minor difference will repeat across training seeds. An infrastructure error makes the policy comparison invalid.


In [ ]:
from vla_lab.evaluation.paired import load_episode_rows, paired_report

pairs = [(0, 1), (1, 2)]
for first, second in pairs:
    a_name = summary["variants"][first]["variant"]
    b_name = summary["variants"][second]["variant"]
    a_rows = load_episode_rows(SUITE_DIR / "variants" / a_name / "evaluation" / "episodes.jsonl")
    b_rows = load_episode_rows(SUITE_DIR / "variants" / b_name / "evaluation" / "episodes.jsonl")
    paired = paired_report(a_rows, b_rows)
    print(a_name, "vs", b_name, "paired difference:", paired["paired_difference"],
          "a-only:", paired["a_only"], "b-only:", paired["b_only"],
          "exact McNemar p:", paired["exact_mcnemar_p"],
          "valid:", paired["valid_for_policy_comparison"])


## 6. Inspect one saved video and report a limit

A live video shows how one chosen checkpoint acted in the simulator. It is not a replay of dataset frames. The source demonstration set may overlap simulator starts; this course has not proved new-placement generalization.


In [ ]:
variant_name = summary["variants"][0]["variant"]
rows_path = SUITE_DIR / "variants" / variant_name / "evaluation" / "episodes.jsonl"
rows = [json.loads(line) for line in rows_path.read_text().splitlines()]
assert [row["initial_state_id"] for row in rows] == list(recipe.state_ids)
failures = [row for row in rows if not row["success"]]
chosen = failures[0] if failures else rows[0]
print("Chosen state:", chosen["initial_state_id"], "success:", chosen["success"],
      "reason:", chosen["termination_reason"])
video = chosen["artifact_paths"].get("video")
if video: display(Video(video, embed=True, html_attributes="controls"))

## Exercise and completion check

**Exercise:** Copy the YAML recipe and change only the smaller variant's demonstration count from 10 to 20. Keep the 39-episode and chunk-size variants, seed, training steps, and state IDs fixed. Run the new suite in a new output directory. Compare the 10, 20, and 39 episode rows and describe one limit of a single-seed result. The [exercise](exercise.md) gives commands; the [solution guide](../../solutions/03-train-act.md) is separate.

You are done when you can name each variant's training episodes and statistics hash, restore a checkpoint, show a finite loss curve, and account for every development state. A failed job retains its log and partial files. A low development score does not justify opening final states to tune the recipe.


In [ ]:
# EXERCISE: Keep this false for the reference path. Use exercise.md to launch a new suite.
RUN_EXERCISE = False
EXERCISE_EPISODE_COUNT = 20
print("New demonstration count:", EXERCISE_EPISODE_COUNT, "Launch requested:", RUN_EXERCISE)